# Chat Template Comparison: Multi-turn & Tool Calling

How different models format conversations via `apply_chat_template`.

In [1]:
from transformers import AutoTokenizer

MODELS = {
    "Gemma 3": "google/gemma-3-4b-it",
    "Qwen3": "Qwen/Qwen3-0.6B",
    "DeepSeek-R1": "deepseek-ai/DeepSeek-R1-0528-Qwen3-8B",
    "Phi-4-reasoning": "microsoft/Phi-4-reasoning",
    "Llama 3.3": "meta-llama/Llama-3.3-70B-Instruct",
}

tokenizers = {}
for label, model_id in MODELS.items():
    try:
        tokenizers[label] = AutoTokenizer.from_pretrained(
            model_id, trust_remote_code=True
        )
        print(f"{label}: loaded")
    except Exception as e:
        print(f"{label}: FAILED ({e})")

Gemma 3: loaded
Qwen3: loaded


config.json:   0%|          | 0.00/859 [00:00<?, ?B/s]

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}


DeepSeek-R1: loaded


config.json:   0%|          | 0.00/796 [00:00<?, ?B/s]

Phi-4-reasoning: loaded
Llama 3.3: loaded


## Multi-turn conversation (no tools)

In [3]:
multi_turn = [
    {"role": "user", "content": "What is 2+2?"},
    {"role": "assistant", "content": "The answer is 4."},
    {"role": "user", "content": "And 3+3?"},
    {"role": "assistant", "content": "The answer is 6."},
]

for label, tok in tokenizers.items():
    print(f"=== {label} ===")
    print(
        tok.apply_chat_template(multi_turn, tokenize=False, add_generation_prompt=False)
    )
    print()

=== Gemma 3 ===
<bos><start_of_turn>user
What is 2+2?<end_of_turn>
<start_of_turn>model
The answer is 4.<end_of_turn>
<start_of_turn>user
And 3+3?<end_of_turn>
<start_of_turn>model
The answer is 6.<end_of_turn>


=== Qwen3 ===
<|im_start|>user
What is 2+2?<|im_end|>
<|im_start|>assistant
The answer is 4.<|im_end|>
<|im_start|>user
And 3+3?<|im_end|>
<|im_start|>assistant
<think>

</think>

The answer is 6.<|im_end|>


=== DeepSeek-R1 ===
<｜begin▁of▁sentence｜><｜User｜>What is 2+2?<｜Assistant｜>The answer is 4.<｜end▁of▁sentence｜><｜User｜>And 3+3?<｜Assistant｜>The answer is 6.<｜end▁of▁sentence｜>

=== Phi-4-reasoning ===
<|im_start|>system<|im_sep|>You are Phi, a language model trained by Microsoft to help users. Your role as an assistant involves thoroughly exploring questions through a systematic thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, reflection, backtraci

## Tool calling conversation

Pattern: user asks -> assistant calls tool -> tool returns result -> assistant answers

In [4]:
tool_conv = [
    {"role": "user", "content": "What is the weather in SF?"},
    {"role": "assistant", "content": ""},
    {"role": "tool", "content": '{"temperature": 72}'},
    {"role": "assistant", "content": "It is 72 degrees."},
]

for label, tok in tokenizers.items():
    print(f"=== {label} ===")
    try:
        print(
            tok.apply_chat_template(
                tool_conv, tokenize=False, add_generation_prompt=False
            )
        )
    except Exception as e:
        print(f"NOT SUPPORTED: {type(e).__name__}: {e}")
    print()

=== Gemma 3 ===
NOT SUPPORTED: TemplateError: Conversation roles must alternate user/assistant/user/assistant/...

=== Qwen3 ===
<|im_start|>user
What is the weather in SF?<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>user
<tool_response>
{"temperature": 72}
</tool_response><|im_end|>
<|im_start|>assistant
<think>

</think>

It is 72 degrees.<|im_end|>


=== DeepSeek-R1 ===
<｜begin▁of▁sentence｜><｜User｜>What is the weather in SF?<｜Assistant｜><｜end▁of▁sentence｜><｜tool▁outputs▁begin｜><｜tool▁output▁begin｜>{"temperature": 72}<｜tool▁output▁end｜><｜tool▁outputs▁end｜>It is 72 degrees.<｜end▁of▁sentence｜>

=== Phi-4-reasoning ===
<|im_start|>system<|im_sep|>You are Phi, a language model trained by Microsoft to help users. Your role as an assistant involves thoroughly exploring questions through a systematic thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, refle

## Qwen3 `<think>` behavior deep dive

Qwen3 injects `<think>...</think>` only on the **last** assistant turn.

In [5]:
tok = tokenizers["Qwen3"]

print("--- Without <think> in content ---")
print(
    tok.apply_chat_template(
        [
            {"role": "user", "content": "Hi"},
            {"role": "assistant", "content": "Hello"},
        ],
        tokenize=False,
        add_generation_prompt=False,
    )
)

print()
print("--- With <think> already in content ---")
print(
    tok.apply_chat_template(
        [
            {"role": "user", "content": "Hi"},
            {
                "role": "assistant",
                "content": "<think>\nLet me think.\n</think>\n\nHello",
            },
        ],
        tokenize=False,
        add_generation_prompt=False,
    )
)

print()
print("--- Multi-turn: all with <think> (template strips earlier turns) ---")
print(
    tok.apply_chat_template(
        [
            {"role": "user", "content": "What is 2+2?"},
            {"role": "assistant", "content": "<think>\nBasic math.\n</think>\n\n4"},
            {"role": "user", "content": "And 3+3?"},
            {"role": "assistant", "content": "<think>\nAlso easy.\n</think>\n\n6"},
        ],
        tokenize=False,
        add_generation_prompt=False,
    )
)

--- Without <think> in content ---
<|im_start|>user
Hi<|im_end|>
<|im_start|>assistant
<think>

</think>

Hello<|im_end|>


--- With <think> already in content ---
<|im_start|>user
Hi<|im_end|>
<|im_start|>assistant
<think>
Let me think.
</think>

Hello<|im_end|>


--- Multi-turn: all with <think> (template strips earlier turns) ---
<|im_start|>user
What is 2+2?<|im_end|>
<|im_start|>assistant
4<|im_end|>
<|im_start|>user
And 3+3?<|im_end|>
<|im_start|>assistant
<think>
Also easy.
</think>

6<|im_end|>



## Key observations

| Model | Tool role | Reasoning injection | Notes |
|-------|-----------|-------------------|-------|
| **Gemma 3** | Not supported | None | Strict user/model alternation |
| **Qwen3** | `tool` -> wrapped as `<tool_response>` under `user` | Injects `<think>` on last turn | Only model that modifies assistant content |
| **DeepSeek-R1** | Custom `tool_output` delimiters | Strips `<think>` from content | Opposite of Qwen3 |
| **Phi-4-reasoning** | `tool` role silently dropped | System prompt instructions only | `<think>` mentioned in system prompt, not injected |
| **Llama 3.3** | `tool` -> `ipython` role | None | Clean separation |